In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from tqdm import tqdm
import math
import os
import re
import tabulate
from IPython.display import display, Markdown
import scipy.optimize as opt

# Custom packages 
import instrumental_programs as iv

# Personalized packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter
from coverage_functions import coverage_calculator, plot_time_series, plot_time_spacing


Preemptively set new Pandas option, also set matplotlib to close

In [ ]:
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

Allow reloading of custom Python classes without resetting kernel

In [ ]:
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_data_merged
%store -r sales_data_merged
%store -r before_after_details_true
%store -r restaurants_by_4m_coverage
%store -r time_differences
%store -r time_differences_details

Read data from parquet files

In [ ]:
# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    %store static_data_merged


# Data already exists
else:
    static_data = static_data_merged.copy()

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    %store sales_data_merged

# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()


# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

In [ ]:
def proportion(loc_id, freq='7D', thinfreq=None, thintype=0):
    # Day of the week: Monday = 0, ..., Sunday = 6
    
    df = sales_and_menu_data[loc_id]
    promo_datetime = before_after_details_true.loc[loc_id, 'cross_over_date']
    two_months_before = promo_datetime - pd.DateOffset(weeks=8)
    two_months_after = promo_datetime + pd.DateOffset(weeks=8)
    df = df[two_months_before:two_months_after]
    
    if thinfreq:
        if thinfreq == 'day':
            df[df.index.dayofweek == thintype]
        elif thinfreq == 'hour':
            df[df.index.hour == thintype]
        else:
            raise Exception('Invalid thintype. Please choose a day of the week or hour of the day.')
    
    # days_offset = (promo_datetime - promo_datetime.floor('W-MON')).days
    # offset_str = f'{days_offset}D'
    
    abf_resampled = (df
                     .query('is_plant_based == "No"')
                     .resample(freq,
                               origin=promo_datetime,
                               #offset=offset_str
                               )
                     ['item_quantity']
                     .sum())
    total_resampled = (df
                     .resample(freq,
                               origin=promo_datetime,
                               #offset=offset_str
                               )
                     ['item_quantity']
                     .sum())
    abf_proportion = abf_resampled / total_resampled
    
    total_resampled_nona = total_resampled[~abf_proportion.isna()]
    abf_proportion_nona = abf_proportion[~abf_proportion.isna()]
    
    return {
        'total_count': total_resampled_nona,
        'abf_proportion': abf_proportion_nona,
        'promo_datetime': promo_datetime,
        'n' : abf_proportion_nona.size
    }


def normal_bins_4(props):
    mean = props.mean()
    sd = props.std()
    if mean - sd < 0 or 1 < mean + sd:
        raise Exception("Variance too high for normal binning.")
    return [0, mean - sd, mean, mean + sd, 1]


def normal_bins_6(props):
    mean = props.mean()
    sd = props.std()
    if mean - 2*sd < 0 or 1 < mean + 2*sd:
        raise Exception("Variance too high for normal binning.")
    return [0, mean - 2*sd, mean - sd, mean, mean + sd, mean + 2*sd, 1]


def max_min_bins_4(props, epsilon=0.01):
    mean = props.mean()
    maxi = props.max() + epsilon
    mini = props.min() - epsilon
    return [mini, (mini+mean)/2, mean, (maxi+mean)/2, maxi]

def normal_half_bins_4(props):
    mean = props.mean()
    sd = props.std()
    if mean - 0.5*sd < 0 or 1 < mean + 0.5*sd:
        raise Exception("Variance too high for half normal binning.")
    return [0, mean - 0.5*sd, mean, mean + 0.5*sd, 1]


def bin(abf_proportion, bins):

    nbins = len(bins) - 1

    raw_categories = np.digitize(abf_proportion, bins)
    
    categories = pd.Categorical(raw_categories, categories=list(range(1,nbins + 1)))

    dated_categories = pd.Series(data=categories, 
                                 index=pd.DatetimeIndex(abf_proportion.index))

    return {
        'abf_category': dated_categories,
        'nbins': nbins,
        'bins': bins,
    }
    
def dist_maker(abf_categories, promo_datetime):
    
    before = abf_categories.loc[:promo_datetime]
    after = abf_categories.loc[promo_datetime:]
    
    before_counts = before.value_counts().sort_index()
    after_counts = after.value_counts().sort_index()
    
    before_props = before_counts / (before_counts.sum() + after_counts.sum())
    after_props = after_counts / (before_counts.sum() + after_counts.sum())
    
    control_props = before_props.values
    treatment_props = after_props.values
    
    return np.array([control_props, treatment_props])

def cost_fun(v):
    return 1*(v<3)

def restaurant_sales_lp_lb(dist, n, nbins, assumptions, alpha=0.05):
    """
    Input dist must match dist_dims
    """
    
    # Setup: define variables for the restaurant problem
    instruments = set()
    measurements = {'T','Y'}
    unobserved = {'Z'}
    graph = [('T', 'Z'),('Z', 'Y')]
    cardinalities = {'T': 2, 'Z': nbins, 'Y': nbins}
    dist_dims = ['T', 'Y']
    
    # Build the linear program
    linprog_args = iv.build_lp(graph,
                               cardinalities,
                               instruments,
                               measurements,
                               unobserved,
                               dist,
                               dist_dims,
                               target_var='Z',
                               intervention={'T': 0},
                               intervention2={'T': 1},
                               assumptions=assumptions,
                               #n=100000,
                               #alpha=alpha,
                               cost_fun=cost_fun
                               )
    return linprog_args

# Measurement error assumptions
assumptions1 = [
    {
        'type': 'symmetry',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {},
    }
]
assumptions2 = [
    {
        'type': 'symmetry',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {},
    },
    {
        'type': 'increasing_errors',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {
            'epsilon': 0.3
            },
    },
]

assumptions3 = [
    {
        'type': 'symmetry',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {},
    },
    {
        'type': 'error_bound',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {
            #'distance':1, 
            #'epsilon': 0.01
            },
    }
]

assumptions4 = [
    {
        'type': 'symmetry',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {},
    },
    {
        'type': 'increasing_errors',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {
            'epsilon': 0.3
            },
    },
    {
        'type': 'error_bound',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {
            #'distance':1, 
            #'epsilon': 0.01
            },
    }
]

One variation

In [ ]:
low_data_loc_ids = ['LQ5EH4BKGV61T']
loc_id = low_data_loc_ids[0]
summary = proportion(loc_id)
counts, props, promo, n = (summary[key] for key in ['total_count', 'abf_proportion', 'promo_datetime', 'n'])

categorized = bin(props, normal_bins_4(props))
categories, nbins, bins_used = (categorized[key] for key in ['abf_category', 'nbins', 'bins'])

dist = dist_maker(categories, promo)

assumptions = [
    {
        'type': 'symmetry',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {},
    },
    # {
    #     'type': 'increasing_errors',
    #     'parent': 'Z',
    #     'child': 'Y',
    #     'kwargs': {
    #         'epsilon': 0.2
    #         },
    # },
    {
        'type': 'error_bound',
        'parent': 'Z',
        'child': 'Y',
        'kwargs': {
            'distance':1, 
            'epsilon': 0.01
            },
    }
]

c, A_ub, b_ub, A_eq, b_eq = restaurant_sales_lp_lb(dist, n, nbins, assumptions)
lb_res = opt.linprog(c, A_ub, b_ub, A_eq, b_eq)
ub_res = opt.linprog(-c, A_ub, b_ub, A_eq, b_eq)
lb_res.fun, - ub_res.fun if ub_res.success else None

All variations

In [ ]:
input_variations = []

# Setup and execute
low_data_loc_ids = ['LQ5EH4BKGV61T']
bin_funcs = [
    normal_half_bins_4, 
    normal_bins_4, 
    normal_bins_6, 
    max_min_bins_4]
assumption_lists = [assumptions1, assumptions2, assumptions3, assumptions4]
epsilon_max = .05
epsilon_count = 6 # includes 0 and epsilon_max, so we need n+1 as the count, e.g., 6 = 5 + 1
epsilons = np.linspace(0, epsilon_max, epsilon_count) 
distance = 1 # the distance bound for error bound assumptions

# Storage
bounds = np.zeros((
    len(low_data_loc_ids), 
    len(bin_funcs), 
    len(assumption_lists), 
    epsilon_count, 
    2 # 2 for upper and lower and bounds respectively
    )) 

dist_info = []
for i, loc_id in enumerate(low_data_loc_ids):
    
    # Retrieve proportions
    summary = proportion(loc_id)
    counts, props, promo, n = (summary[key] for key in ['total_count', 'abf_proportion', 'promo_datetime', 'n'])

    # Choose bins
    bin_lists = []
    for bin_func in bin_funcs:
        bin_lists.append(bin_func(props))

    # Analyze every binning scheme
    for j, bin_list in enumerate(bin_lists):
    
        # Retrieve categories and make a distribution
        categorized = bin(props, bin_list)
        categories, nbins, bins_used = (categorized[key] for key in ['abf_category', 'nbins', 'bins'])
        dist = dist_maker(categories, promo)
        
        for k, assumption_list in enumerate(assumption_lists):
        
            if k == 0 or k == 1:
                
                input_variations.append((loc_id, dist, n, nbins, assumption_list, -1))
            
            else:
                
                for l, epsilon in tqdm(enumerate(epsilons)):
                    
                    # Modify epsilon for relevant assumption sets
                    for a, assumption in enumerate(assumption_list):
                        assumption_c = assumption.copy()
                        # if assumption_c['type'] == 'increasing_errors':
                        #     assumption_c['kwargs'] = {'epsilon': epsilon}
                        if assumption_c['type'] == 'error_bound':
                            assumption_c['kwargs'] = {'distance': distance, 'epsilon': epsilon}
                        assumption_list[a] = assumption_c

                    input_variations.append((loc_id, dist, n, nbins, assumption_list.copy(), epsilon))

        # Store
        dist_info.append({'loc_id': loc_id,
                          'bins': bin_list,
                          'categories': categories,
                          'dist': dist})

In [ ]:
# [row['dist'] for row in dist_info]

max_cols = max(dist.shape[1] for dist in [row['dist'] for row in dist_info])
all_dists = []
multiindex = []
for idx, info in enumerate(dist_info):
    loc_id = info['loc_id']
    categories = info['categories']
    dist = info['dist']
    
    # Pad arrays to the maximum column size
    padded_dist = np.pad(dist, ((0, 0), (0, max_cols - dist.shape[1])), constant_values=np.nan)
    all_dists.append(padded_dist)
    
    bin_name = f'bin{idx + 1}'
    for i in range(padded_dist.shape[0]):
        multiindex.append((bin_name, i % 2))  # Alternates between 0 and 1
    
        
index = pd.MultiIndex.from_tuples(multiindex, names=["loc_id", "inner_index"])

# Create DataFrame
stacked_df = pd.DataFrame(np.vstack(all_dists), index=index, columns=[f'{i+1}' for i in range(max_cols)])

stacked_df

In [ ]:
total_results = []
for loc_id, dist, n, nbins, assumption_list, epsilon in tqdm(input_variations):
    c, A_ub, b_ub, A_eq, b_eq = restaurant_sales_lp_lb(dist, n, nbins, assumption_list)
    lb_res = opt.linprog(c, A_ub, b_ub, A_eq, b_eq)
    ub_res = opt.linprog(-c, A_ub, b_ub, A_eq, b_eq)
    # bounds[i, j, k, l, 0] = lb_res.fun
    # bounds[i, j, k, l, 1] = -ub_res.fun
    if lb_res.success and ub_res.success:
        total_results.append((loc_id, dist, n, nbins, assumption_list, epsilon, lb_res.fun, -ub_res.fun))
    else:
        print(f"Solver failed for epsilon={epsilon} with messages: lb_res={lb_res.message}, ub_res={ub_res.message}")
        total_results.append((loc_id, dist, n, nbins, assumption_list, epsilon, None, None))

In [ ]:
# total_results
# analysis_results = pd.concat([bounding_results_take_2_first_two_thirds[0:14][4], bounding_results_take_2_first_two_thirds[0:14][[6,7]], bounding_results_take_2_first_two_thirds[14:28].reset_index()[[6,7]]], axis=1)
# analysis_results[4] = [(assump[2]['type'], assump[2]['kwargs'], assump[1]['type'], assump[1]['kwargs']) if 2 < len(assump) else ((assump[1]['type'], assump[1]['kwargs']) if 1 < len(assump) else assump[0]['type']) for assump in analysis_results[4].tolist()]
# analysis_results.columns = ['Assumption', 'Lower Bound (Half)', 'Upper Bound (Half)', 'Lower (Full)', 'Upper Bound (Full)']
# analysis_results['Assumption'] = analysis_results['Assumption'].astype(str).str.replace('{|}|silon|ance|metry','', regex=True).str.replace("'error_bound',",'eb:', regex=True).str.replace("'increasing_errors',",'ie:', regex=True)

%store -r analysis_results

In [ ]:
analysis_results

In [ ]:
just_err_bounds = analysis_results.loc[2:7].assign(Epsilon = np.linspace(0,0.05,6))
err_bounds_and_inc_err = analysis_results.loc[8:14].assign(Epsilon = np.linspace(0,0.05,6))
plt.figure(figsize=(4.8, 3.6))
plt.fill_between(just_err_bounds.Epsilon, 
                 just_err_bounds['Lower Bound (Half)'], 
                 just_err_bounds['Upper Bound (Half)'], 
                 color='darkblue', alpha=1, label='EB (HalfSD Binning)')
plt.fill_between(just_err_bounds.Epsilon, 
                 just_err_bounds['Lower (Full)'], 
                 just_err_bounds['Upper Bound (Full)'], 
                 color='darkblue', alpha=0.5, label='EB (SD and MinMax Binning)')

plt.fill_between(err_bounds_and_inc_err.Epsilon, 
                 err_bounds_and_inc_err['Lower Bound (Half)'], 
                 err_bounds_and_inc_err['Upper Bound (Half)'], 
                 color='limegreen', alpha=1, label='EB+Mon (HalfSD Binning)')
plt.fill_between(err_bounds_and_inc_err.Epsilon, 
                 err_bounds_and_inc_err['Lower (Full)'], 
                 err_bounds_and_inc_err['Upper Bound (Full)'], 
                 color='lightgreen', alpha=1, label='EB+Mon (SD and MinMax Binning)')

plt.legend(title='Legend', loc='right', fontsize=7, title_fontsize=9)
#plt.title('Measurement Error Bounds for \nP(Z(T=1)<3) - P(Z(T=0)<3)', fontsize=10)
plt.ylabel('ATE', fontsize=10)
plt.xlabel(r'Error budget ($\varepsilon$)', fontsize=10)
plt.ylim(-1,1)
plt.yticks(fontsize=7)
plt.xticks(fontsize=7)
plt.savefig('errorbudget.png', dpi=600)
plt.show()

$\varepsilon$

In [ ]:
# import pickle
# from multiprocessing import Pool
# import copy

# # Test serialization
# for variation in input_variations:
#     try:
#         pickle.dumps(variation)
#     except Exception as e:
#         print(f"Serialization failed for input: {variation}, error: {e}")    

# # Ensure assumptions are serializable
# def make_assumptions_serializable(assumptions):
#     return [copy.deepcopy(assumption) for assumption in assumptions]

# # Update the input variations (removing loc_id if unused)
# input_variations = [
#     (dist, n, nbins, make_assumptions_serializable(assumption_list)) for loc_id, dist, n, nbins, assumption_list in input_variations
# ]

# # Define the worker function
# def linear_program_runner(args):
#     dist, n, nbins, assumptions = args
#     try:
#         c, A_ub, b_ub, A_eq, b_eq = restaurant_sales_lp_lb(dist, n, nbins, assumptions)
#         lb_res = opt.linprog(c, A_ub, b_ub, A_eq, b_eq, method="highs")
#         ub_res = opt.linprog(-c, A_ub, b_ub, A_eq, b_eq, method="highs")
#         return lb_res.fun, -ub_res.fun
#     except Exception as e:
#         print(f"Error in linear_program_runner with args={args}: {e}")
#         return None, None  # Or log the error if necessary

# # Use multiprocessing.Pool
# if __name__ == "__main__":
#     # Optional: Set multiprocessing start method
#     import multiprocessing
#     if multiprocessing.get_start_method(allow_none=True) is None:
#         multiprocessing.set_start_method("spawn")

#     with Pool(processes=4) as pool:  # Adjust number of processes as needed
#         results = pool.map(linear_program_runner, input_variations[:1])

#     # Process results
#     result_index = 0
#     for i, loc_id in enumerate(low_data_loc_ids):
#         for j, bin_list in enumerate(bin_lists):
#             for k, assumption_list in enumerate(assumption_lists):
#                 for l, epsilon in enumerate(epsilons):
#                     if results[result_index] is not None:
#                         lb, ub = results[result_index]
#                         bounds[i, j, k, l, 0] = lb
#                         bounds[i, j, k, l, 1] = ub
#                     else:
#                         print(f"Failed computation at index {result_index}")
#                     result_index += 1